# Taller: Centro de Colombia (Algoritmo Naïve)

Este notebook está diseñado para encontrar el círculo mínimo contenedor (MEC - *Minimum Enclosing Circle*) de los municipios de Colombia y condados de EE. UU. utilizando un **algoritmo naïve** y **aritmética compleja**.

## Representación de puntos con números complejos
Cada municipio/condado se representa como un número complejo:
$$z = \text{longitud} + i \cdot \text{latitud}$$

## Fórmulas de Aritmética Compleja

### 1. Círculo a partir de 2 puntos ($z_1$ y $z_2$)
Si el círculo está definido por el segmento que une dos puntos como su diámetro:
- **Centro ($c$):** El punto medio entre ambos.
  $$c = \frac{z_1 + z_2}{2}$$
- **Radio ($r$):** La mitad de la distancia entre ambos.
  $$r = \frac{|z_1 - z_2|}{2}$$

### 2. Círculo a partir de 3 puntos ($z_1$, $z_2$, y $z_3$)
Para hallar el circuncentro ($c$) y circunradio ($r$) de tres puntos no colineales, trasladamos el origen a $z_1$, definiendo:
$$A = z_2 - z_1 \quad y \quad B = z_3 - z_1$$
El circuncentro trasladado $W$ satisface el sistema:
$$\bar{A}W + A\bar{W} = |A|^2$$
$$\bar{B}W + B\bar{W} = |B|^2$$
Resolviendo para $W$ en términos de operaciones complejas:
$$W = \frac{B|A|^2 - A|B|^2}{B\bar{A} - A\bar{B}}$$
Finalmente, deshacemos la traslación para encontrar el centro real y el radio:
$$c = z_1 + W$$
$$r = |c - z_1|$$

*Nota: Si el denominador $D = B\bar{A} - A\bar{B} = 2i \cdot \text{Im}(B\bar{A})$ es cercano a cero, significa que los puntos son colineales y no definen un círculo único.*

## 1. Importación y Limpieza de Datos

La siguiente celda carga los datos de ambos archivos de Excel (`DIVIPOLA_Municipios.xlsx` y `uscounties.xlsx`), filtra los valores nulos, extrae la latitud y longitud, y crea arreglos de números complejos `colombia_points` y `usa_points` para tus cálculos.

In [ ]:
import pandas as pd
import numpy as np

# --- 1. CARGA Y LIMPIEZA DE DATOS DE COLOMBIA ---
def cargar_colombia():
    path_col = 'Listo para entregar/DIVIPOLA_Municipios.xlsx'
    # Omitimos las primeras 9 filas donde están los logos y encabezados decorativos
    df_col = pd.read_excel(path_col, skiprows=9)
    # Asignamos nombres limpios a las columnas relevantes
    df_col.columns = ['dpt_code', 'dpt_name', 'mun_code', 'mun_name', 'type', 'lat', 'lon', 'extra']
    
    # Eliminamos filas vacías o con cabeceras repetidas
    df_col = df_col.dropna(subset=['lat', 'lon'])
    df_col = df_col[df_col['lat'] != 'LATITUD']
    
    # Convertimos coordenadas a tipo flotante
    df_col['lat'] = df_col['lat'].astype(float)
    df_col['lon'] = df_col['lon'].astype(float)
    
    # Creamos el arreglo de complejos (z = lon + i * lat)
    z_colombia = df_col['lon'].values + 1j * df_col['lat'].values
    print(f"Colombia: Cargados {len(z_colombia)} municipios correctamente.")
    return z_colombia, df_col

# --- 2. CARGA Y LIMPIEZA DE DATOS DE USA ---
def cargar_usa():
    path_usa = 'Listo para entregar/uscounties.xlsx'
    df_usa = pd.read_excel(path_usa)
    
    df_usa = df_usa.dropna(subset=['lat', 'lng'])
    df_usa['lat'] = df_usa['lat'].astype(float)
    df_usa['lng'] = df_usa['lng'].astype(float)
    
    # Creamos el arreglo de complejos (z = lng + i * lat)
    z_usa = df_usa['lng'].values + 1j * df_usa['lat'].values
    print(f"USA: Cargados {len(z_usa)} condados correctamente.")
    return z_usa, df_usa

colombia_points, df_colombia = cargar_colombia()
usa_points, df_usa = cargar_usa()

## 2. Guía para Implementar el Algoritmo Naïve del Círculo Mínimo Contenedor (MEC)

El algoritmo naïve de fuerza bruta para encontrar el círculo mínimo contenedor consiste en encontrar un círculo que contenga **todos** los puntos y cuyo radio sea el **mínimo posible**.

Matemáticamente, el MEC está determinado de manera única por:
1. **Un par de puntos** que forman el diámetro del círculo.
2. **Una terna de puntos** que forman el círculo circunscrito de un triángulo acutángulo.

### Pasos para tu implementación:

1. **Definir una función para calcular el círculo de 3 puntos**:
   Escribe una función que tome `z1, z2, z3` y devuelva el centro `c` (complejo) y el radio `r` (real) usando la fórmula de la sección anterior.
   
2. **Definir una función de verificación**:
   Una función que tome un centro `c`, un radio `r` y la lista completa de puntos, y valide si todos los puntos están dentro del círculo, es decir, si para todo punto $p$, $|p - c| \le r + \epsilon$ (usa una tolerancia $\epsilon \approx 10^{-9}$ para evitar errores de precisión de punto flotante).

3. **Iterar sobre todas las Parejas**:
   - Genera el círculo para cada pareja de puntos $(z_i, z_j)$.
   - Verifica si contiene a todos los demás puntos.
   - Si es así, guarda este círculo si su radio es menor al mínimo encontrado hasta ahora.

4. **Iterar sobre todas las Ternas**:
   - Genera el círculo para cada terna de puntos $(z_i, z_j, z_k)$.
   - Verifica si contiene a todos los demás puntos.
   - Si es así, guarda este círculo si su radio es menor al mínimo encontrado hasta ahora.

### 💡 Tips de Optimización (muy importantes para no esperar horas):
- **Rompe el bucle de verificación de inmediato**: Cuando estés comprobando si todos los puntos están dentro de un círculo candidato y encuentres el **primer** punto que está afuera, haz un `break` inmediatamente. No tiene sentido seguir revisando los demás puntos.
- **Ordena los puntos desde el exterior**: Si ordenas los puntos de mayor a menor distancia respecto al centroide de la nube de puntos, la verificación fallará casi siempre en los primeros 2 o 3 puntos, haciendo que el chequeo de exclusión sea extremadamente rápido ($O(1)$ en promedio).
- **Poda por radio mínimo**: Si calculas el radio de un par o terna y ves que $r \ge R_{min}$ (donde $R_{min}$ es el radio del círculo contenedor válido más pequeño que has hallado hasta ahora), descarta ese círculo inmediatamente sin hacer ninguna verificación de pertenencia.

In [ ]:
# --- 3. IMPLEMENTACIÓN DEL ALGORITMO NAÏVE (SIN OPTIMIZAR) ---

def obtener_circuncentro(z1, z2, z3):
    """
    Calcula el circuncentro (c) y circunradio (r) de tres puntos complejos.
    Devuelve (None, None) si los puntos son colineales.
    """
    A = z2 - z1
    B = z3 - z1
    num = B * (A * A.conjugate()) - A * (B * B.conjugate())
    den = B * A.conjugate() - A * B.conjugate()
    
    if abs(den) < 1e-12:
        return None, None
    
    W = num / den
    c = z1 + W
    r = abs(c - z1)
    return c, r

def contiene_todos(c, r, puntos):
    """
    Verifica si todos los puntos están dentro del círculo definido por (c, r).
    Usa una tolerancia para evitar errores de precisión numérica.
    """
    tolerancia = 1e-9
    for p in puntos:
        if abs(p - c) > r + tolerancia:
            return False
    return True

def encontrar_mec_naive(puntos):
    """
    Busca el círculo mínimo contenedor (MEC) recorriendo exhaustivamente
    todas las parejas y ternas posibles.
    """
    n = len(puntos)
    R_min = float('inf')
    c_best = None
    puntos_frontera = []

    # 1. Probar todas las Parejas (Círculos definidos por 2 puntos)
    for i in range(n):
        for j in range(i + 1, n):
            z1, z2 = puntos[i], puntos[j]
            c = (z1 + z2) / 2.0
            r = abs(z1 - z2) / 2.0
            
            if contiene_todos(c, r, puntos):
                if r < R_min:
                    R_min = r
                    c_best = c
                    puntos_frontera = [z1, z2]

    # 2. Probar todas las Ternas (Círculos definidos por 3 puntos)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                z1, z2, z3 = puntos[i], puntos[j], puntos[k]
                c, r = obtener_circuncentro(z1, z2, z3)
                
                if r is not None:
                    if contiene_todos(c, r, puntos):
                        if r < R_min:
                            R_min = r
                            c_best = c
                            puntos_frontera = [z1, z2, z3]
                            
    return c_best, R_min, puntos_frontera

### ⚠️ Advertencia sobre la Complejidad Temporal

Este algoritmo naive explora todas las combinaciones:
- Parejas: $\binom{N}{2} \approx O(N^2)$ operaciones de validación.
- Ternas: $\binom{N}{3} \approx O(N^3)$ combinaciones. Cada una requiere revisar los $N$ puntos, resultando en un costo total de **$O(N^4)$**.

Para **Colombia** ($N = 1122$), la ejecución completa tomaría aproximadamente $2.35 \times 10^8$ evaluaciones en el bucle principal, lo cual puede tardar **varias horas** en Python estándar.
Para **EE. UU.** ($N = 3144$), tomaría más de $5 \times 10^9$ evaluaciones.

#### Celda de Prueba (Subconjunto Pequeño)
Para validar que tu algoritmo funciona correctamente sin congelar el notebook, puedes correrlo primero con un subconjunto (por ejemplo, los primeros 50 o 100 municipios):

In [ ]:
import time

# Tomamos una muestra de los primeros 50 municipios para una prueba rápida
puntos_prueba = colombia_points[:50]

t0 = time.time()
centro, radio, frontera = encontrar_mec_naive(puntos_prueba)
t1 = time.time()

print(f"Prueba completada en: {t1 - t0:.4f} segundos.")
print(f"Centro encontrado: {centro}")
print(f"Radio: {radio}")
print(f"Puntos en la frontera (complejos): {frontera}")

#### Ejecución Completa (Bajo tu propio riesgo ⏱️)
Descomenta las líneas de abajo si deseas ejecutar el algoritmo con todos los puntos de Colombia:

In [ ]:
# t0 = time.time()
# centro_col, radio_col, frontera_col = encontrar_mec_naive(colombia_points)
# t1 = time.time()
# print(f"Colombia completa resuelta en: {t1 - t0:.4f} segundos.")